# Diffuse-defect inventory (FF-HEDM) — end-to-end

`midas_defect` quantifies the diffuse field *around and between* the Bragg peaks of an
indexed FF-HEDM dataset to produce a per-grain defect inventory:

**geometry QC → Bragg/diffuse split → 100 % intensity budget → dislocation density →
forbidden-reflection test → ⟨111⟩ fault rods → fault probability**

This notebook is **phase-agnostic**: edit the single *dataset* cell to point at another
sample (e.g. a genuine CuAl₂ superstructure dataset) — swap `fcc_cu_crystal()` for
`cual2_crystal()` and the geometry/data paths. Nothing downstream changes.

In [ ]:
import numpy as np
from midas_defect.lattice import fcc_cu_crystal, cual2_crystal
from midas_defect.examples.demk_fcc_end_to_end import (
    DemkGeometry, voxels_to_qsample,
)
from midas_defect.bragg_diffuse import predicted_reflection_points, classify_voxels, on_lattice_fraction
from midas_defect.intensity_budget import intensity_budget
from midas_defect.williamson_hall import dislocation_density_per_grain
from midas_defect.defect_tests import forbidden_reflection_test, fault_rod_alignment, fault_probability_alpha

## 1. Choose the dataset and phase

**This is the only cell to edit when re-pointing to another sample.** For a genuine
CuAl₂ dataset, set `crystal = cual2_crystal()` and update `geom`/paths.

In [ ]:
crystal = fcc_cu_crystal()          # demk default; for CuAl2 use cual2_crystal()
geom    = DemkGeometry()            # validated gold-calibrant detector model

# sparse voxel cloud (indices=[frame,row,col] + values) and the indexed grains
VOXELS = "/path/to/voxels_layerXXXX.npz"   # e.g. binned NPZ or full-res export
GRAINS = "/path/to/LayerNr_1/Grains.csv"   # MIDAS Grains.csv (orientation matrices)

## 2. Load → convert to sample-frame q

`voxels_to_qsample` reuses `midas_transforms.apply_tilt_distortion` (the validated,
distortion-aware detector model) and the ω = 180 − 0.25·frame map.

In [ ]:
d = np.load(VOXELS, allow_pickle=False)
det_bin = int(d["det_bin"])
qs  = voxels_to_qsample(d["indices"], det_bin, geom)
val = d["values"].astype(np.float64)
OM  = np.genfromtxt(GRAINS, comments="%")[:, 1:10].reshape(-1, 3, 3)
qmag = np.linalg.norm(qs, axis=1)
print(f"{len(val):,} voxels | {len(OM)} grains")

## 3. Geometry QC + Bragg/diffuse split + intensity budget

A correct geometry puts ~96 % of bright voxels on the predicted lattice; the budget
must close to 100 %.

In [ ]:
P = predicted_reflection_points(OM, crystal, q_max_inv_A=8.5).numpy()
olf = on_lattice_fraction(qs, val, P, bright_percentile=99.5, tol_inv_A=0.1)
print(f"geometry QC: {100*olf:.1f}% of bright voxels on lattice")

split  = classify_voxels(qs, val, P, tol_inv_A=0.05)
budget = intensity_budget(split.dist_to_lattice, qmag, val)
print(budget)

## 4. Defect inventory

Dislocation density (Williamson–Hall on Bragg radial breadth), the forbidden-reflection
(selection-rule) test, the explicit ⟨111⟩ fault-rod test, and the fault-probability proxy.

In [ ]:
wh   = dislocation_density_per_grain(qs, val, OM, crystal)
forb = forbidden_reflection_test(qs, val, OM, crystal)
frod = fault_rod_alignment(qs, val, OM, crystal)
fa   = fault_probability_alpha(qs, val, OM, crystal)

print(f"dislocation density (b={wh.burgers_A:.3f} A): median {wh.rho_median_per_m2:.2e} m^-2 "
      f"(D~{wh.domain_size_A_median:.0f} A, {wh.n_grains_fit}/{wh.n_grains} grains)")
print(f"forbidden excess: {forb.excess_median:+.4f} ({forb.n_grains_excess}/{forb.n_grains} grains)")
print(f"<111> fault rods (along/perp): median {frod.along_over_perp_median:.2f}, "
      f"{100*frod.frac_grains_enriched:.0f}% of grains > 1.2x")
print(f"fault alpha: median {fa.alpha_median:.4f} (faulted third {fa.faulted_third_median:.4f})")

## 5. Re-pointing to a different phase / sample

Go back to **cell 1** and set `crystal = cual2_crystal()` (tetragonal θ-Al₂Cu) plus the
appropriate `geom` and data paths. The selection-rule, rod, and budget machinery are all
driven by the `Crystal`, so the rest of the notebook runs unchanged — this is the path for
analysing a genuine CuAl₂ superstructure dataset.